# Explore the actual GLiNER training dataset

**1,898 rule-labeled entity spans, 25 documents, 62 chunks, 14 labels.** This notebook loads the public release without credentials, validates both viewer representations, inspects class imbalance and checks source/split integrity. It does not train a model or call an LLM. Spans are not independent manual samples. The related-document test does not measure patent accuracy.

In [ ]:
%pip install -q datasets==5.0.1 huggingface-hub==1.33.0 matplotlib==3.11.2

In [ ]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from collections import Counter
import hashlib, json
from pathlib import Path
import matplotlib.pyplot as plt

REPO = "cmuchancel/gliner-sysml-training-data"
chunks = load_dataset(REPO, "chunks")
entities = load_dataset(REPO, "entities")
expected = {"train": (54, 1721, 21), "validation": (5, 139, 2), "test": (3, 38, 2)}
seen = set()
for split, (n_chunks, n_entities, n_docs) in expected.items():
    assert len(chunks[split]) == n_chunks and len(entities[split]) == n_entities
    ids = set(chunks[split]["document_id"])
    assert len(ids) == n_docs and not ids & seen
    seen |= ids
    assert sum(len(row["entities"]) for row in chunks[split]) == n_entities
    for row in chunks[split]:
        assert len(row["tokens"]) <= 384
        for e in row["entities"]:
            assert " ".join(row["tokens"][e["start_token"]:e["end_token"]+1]) == e["text"]
    print(split, n_docs, "documents;", n_chunks, "chunks;", n_entities, "labels")
assert len(seen) == 25
chunks["train"][0]

In [ ]:
counts = Counter()
for split in entities:
    counts.update(entities[split]["label"])
assert sum(counts.values()) == 1898 and len(counts) == 14
labels = sorted(counts, key=counts.get)
fig, ax = plt.subplots(figsize=(10, 6))
left = [0] * len(labels)
for split in ("train", "validation", "test"):
    by_label = Counter(entities[split]["label"])
    values = [by_label[label] for label in labels]
    ax.barh(labels, values, left=left, label=split)
    left = [a+b for a,b in zip(left,values)]
ax.set_xlabel("Rule-labeled entity occurrences")
ax.set_title("GLiNER corpus: 1,898 labels")
ax.legend(); fig.tight_layout(); plt.show()

In [ ]:
source = Path(hf_hub_download(REPO, "original/canonical/documents.json", repo_type="dataset", token=False))
documents = json.loads(source.read_text())
for d in documents:
    assert hashlib.sha256(d["text"].encode("utf-8")).hexdigest() == d["sha256"]
    for e in d["entities"]:
        assert d["text"][e["start"]:e["end"]] == e["text"]
test = Path(hf_hub_download(REPO, "original/splits/test.json", repo_type="dataset", token=False))
assert hashlib.sha256(test.read_bytes()).hexdigest() == "2c49631c4d8a60ee965c473a63df3b781aca059657063c22890437fb43204e5f"
print("Original spans and released-checkpoint test checksum verified.")

## Interpretation

Labels were generated by translator rules. Document IDs are disjoint; related actuator variants still cross splits. The test has 38 entities from four labels, with ten unsupported label types. See the [dataset card](https://huggingface.co/datasets/cmuchancel/gliner-sysml-training-data) and [model card](https://huggingface.co/spaces/cmuchancel/patent2sysml/blob/main/model_cards/GLINER.md). No synthetic expansion or manual-label count is asserted. Training is deliberately excluded from this exploration notebook.